In [ ]:
from collections import defaultdict
from matplotlib import pyplot as plt
import torch
from torch import nn
import torchrl
import torchrl.envs as torch_envs
from tqdm import tqdm
import gymnasium as gym
import tensordict
from tensordict import nn as dict_nn
import torchsummary
import torchvision

from spaceship_env import SpaceshipEnv

In [ ]:
device = torch.device("cuda")
lr = 3e-4
max_grad_norm = 1.0

frames_per_batch = 5000
total_frames = 100_000_000

sub_batch_size = 512
num_epochs = 10
clip_epsilon = 0.2
gamma = 0.99
lmbda = 0.95
entropy_eps = 1e-4

In [ ]:
# Spaceship Environment

def make_norm_transforms(env: gym.Env):
    transforms = []
    for key, space in env.observation_space.items():
        if key in ["position", "target", "velocity", "rotation"]:
            transforms.append(torch_envs.transforms.ObservationNorm(loc=space.low, scale=1 / (space.high-space.low), in_keys=key, out_keys=key, standard_normal=False))
    return torch_envs.transforms.Compose(*transforms)

gym.register('Spaceship_Target', entry_point="spaceship_env:SpaceshipEnv")

env = torch_envs.GymEnv('Spaceship_Target', device=device)
print(env.observation_spec.keys())
env = torch_envs.transforms.TransformedEnv(base_env=env, 
                                             transform=torch_envs.Compose([
                                                 make_norm_transforms(env),
                                                 torch_envs.transforms.CatTensors(["position", "target", "velocity", "rotation"], "observation")
                                                 ]))

logged_env = torch_envs.GymEnv('Spaceship_Target', device=torch.device('cuda'), return_pixels=True)
logged_env = torch_envs.transforms.TransformedEnv(base_env=logged_env, 
                                             transform=torch_envs.Compose([
                                                 make_norm_transforms(env),
                                                 torch_envs.transforms.CatTensors(["position", "target", "velocity", "rotation"], "observation")
                                                 ]))

print(logged_env.observation_spec.keys())

print(env.rollout(3)['observation'])
        

In [ ]:
# Bipedal Walker Env

from torchrl.record import PixelRenderTransform

env = torch_envs.GymEnv("BipedalWalker-v3", device=device)

logged_env = torch_envs.GymEnv("BipedalWalker-v3", render_mode='rgb_array', device=device)
logged_env = torch_envs.TransformedEnv(logged_env, PixelRenderTransform('pixels'))

logged_env.observation_spec


In [ ]:
# # Mujoco Humanoid Env

# env = torch_envs.GymEnv("Humanoid-v5", device=device)

# logged_env = torch_envs.GymEnv("Humanoid-v5", render_mode='rgb_array', device=device)
# logged_env = torch_envs.TransformedEnv(logged_env, PixelRenderTransform('pixels'))

# logged_env.observation_spec

In [ ]:
env.action_spec.shape

In [ ]:
from torchrl.modules.tensordict_module import ProbabilisticActor

input_shape = env.observation_spec['observation'].shape[0]
output_shape = env.action_spec.shape[0]

actor = nn.Sequential(
    nn.Linear(input_shape, 24, device=device),
    nn.Tanh(),
    nn.Linear(24, 32, device=device),
    nn.Tanh(),
    nn.Linear(32, 24, device=device),
    nn.Tanh(),
    nn.Linear(24, output_shape, device=device),
)

policy_module = dict_nn.TensorDictModule(actor, in_keys=["observation"], out_keys=["logits"])
policy_module = ProbabilisticActor(module=policy_module,
                                    spec=env.action_spec,
                                    in_keys=["logits"],
                                    distribution_class=torch.distributions.OneHotCategorical,
                                    return_log_prob=True)

In [ ]:
import torchsummary
torchsummary.summary(actor, (input_shape,))

In [ ]:
from torchrl.modules import ValueOperator

value_net = nn.Sequential(
    nn.Linear(env.observation_spec['observation'].shape[0], 24, device=device),
    nn.Tanh(),
    nn.Linear(24, 24, device=device),
    nn.Tanh(),
    nn.Linear(24, 1, device=device),
)

value_module = ValueOperator(value_net, in_keys=["observation"])

In [ ]:
print(policy_module(env.reset())['action'])
print(value_module(env.reset()))

In [ ]:
import torch.nn.functional as F
def simple_rollout(env, action_index, steps):
    td = env.reset()
    results = []
    
    # Get the number of classes from the environment spec
    # usually env.action_spec.shape[-1] for OneHot specs
    n_actions = env.action_spec.shape[-1]
    
    # Create the One-Hot tensor: e.g., 2 -> [0, 0, 1]
    action_one_hot = F.one_hot(torch.tensor(action_index), n_actions)
    
    for _ in range(steps):
        td['action'] = action_one_hot
        td = env.step(td)
        
        # Note: To print the index (2) instead of the vector, use argmax
        print(f"Action: {td['action'].argmax().item()} | Reward: {td[('next', 'reward')].item()}")
        
        results.append(td.clone())
        td = env.step_mdp(td)
    return torch.stack(results)

simple_rollout(env, torch.tensor(2), 100)

In [ ]:
from torchrl.objectives.value import GAE
from torchrl.objectives import ClipPPOLoss

advantage_module = GAE(gamma=gamma, lmbda=lmbda, value_network=value_module, average_gae=True, device=device)
loss_module = ClipPPOLoss(actor_network=policy_module, critic_network=value_module, clip_epsilon=clip_epsilon, entropy_coeff=entropy_eps)
optim = torch.optim.Adam(loss_module.parameters(), lr)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optim, total_frames // frames_per_batch, 0.0)

In [ ]:
from torchrl.collectors import SyncDataCollector
from torchrl.data import ReplayBuffer, LazyTensorStorage, SamplerWithoutReplacement

collector = SyncDataCollector(env, policy_module, frames_per_batch=frames_per_batch, total_frames=total_frames, device=device)
replay_buffer = ReplayBuffer(storage=LazyTensorStorage(max_size=frames_per_batch), sampler=SamplerWithoutReplacement())

In [ ]:
from torchrl.record import VideoRecorder
from torchrl.record.loggers.csv import CSVLogger

logger = CSVLogger(exp_name="Humanoid", log_dir="HumanoidVideos", video_format="mp4")
logged_env = torch_envs.transforms.TransformedEnv(logged_env, VideoRecorder(logger, tag="run_video", in_keys=['pixels']))  # should just use render

In [ ]:
out = logged_env.rollout(1000, lambda *args: env.action_spec.sample())
logged_env.transform[-1].dump()

In [ ]:
type(env.render())

In [ ]:
for data in collector:
    print(data)
    break

In [ ]:
from tensordict.nn import set_interaction_type, InteractionType

logs = defaultdict(list)
pbar = tqdm(total=total_frames)
eval_str = ""

for i, tensordict_data in enumerate(collector):
    for _ in range(num_epochs):
        advantage_module(tensordict_data)
        data_view = tensordict_data.reshape(-1)
        replay_buffer.extend(data_view.cpu()) # not exactly sure why cpu
        for _ in range(frames_per_batch // sub_batch_size):
            subdata = replay_buffer.sample(sub_batch_size)
            loss_vals = loss_module(subdata.to(device))
            loss_value = loss_vals["loss_objective"] + loss_vals["loss_critic"] + loss_vals["loss_entropy"]
            loss_value.backward()
            torch.nn.utils.clip_grad_norm_(loss_module.parameters(), max_grad_norm)
            optim.step()
            optim.zero_grad()
            
        logs["reward"].append(tensordict_data["next", "reward"].mean().item())
        pbar.update(tensordict_data.numel())
        
    cum_reward_str = (
        f"average reward={logs['reward'][-1]: 4.4f} (init={logs['reward'][0]: 4.4f})"
    )
    # logs["step_count"].append(tensordict_data["step_count"].max().item())
    # stepcount_str = f"step count (max): {logs['step_count'][-1]}"
    logs["lr"].append(optim.param_groups[0]["lr"])
    lr_str = f"lr policy: {logs['lr'][-1]: 4.4f}"
    
    if i % 3 == 0:
        with set_interaction_type(InteractionType.DETERMINISTIC), torch.no_grad():
            eval_rollout = logged_env.rollout(600, policy_module)
            logged_env.transform[-1].dump()
            logs["eval reward"].append(eval_rollout["next", "reward"].mean().item())
            logs["eval reward (sum)"].append(
                eval_rollout["next", "reward"].sum().item()
            )
            # logs["eval step_count"].append(eval_rollout["step_count"].max().item())
            eval_str = (
                f"eval cumulative reward: {logs['eval reward (sum)'][-1]: 4.4f} "
                f"(init: {logs['eval reward (sum)'][0]: 4.4f}), "
                # f"eval step-count: {logs['eval step_count'][-1]}"
            )
            del eval_rollout
            
    # pbar.set_description(", ".join([eval_str, cum_reward_str, stepcount_str, lr_str]))
    scheduler.step()


In [ ]:
plt.figure(figsize=(10, 10))
plt.subplot(2, 2, 1)
plt.plot(logs["reward"])
plt.title("training rewards (average)")
plt.subplot(2, 2, 2)
plt.plot(logs["step_count"])
plt.title("Max step count (training)")
plt.subplot(2, 2, 3)
plt.plot(logs["eval reward (sum)"])
plt.title("Return (test)")
plt.subplot(2, 2, 4)
plt.plot(logs["eval step_count"])
plt.title("Max step count (test)")
plt.show()